In [ ]:
import cv2
import mediapipe as mp
import time
from datetime import datetime
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub

# -------------------- MediaPipe Setup --------------------
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose

face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
pose = mp_pose.Pose(
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

# -------------------- EfficientDet Model --------------------
model_url = "https://tfhub.dev/tensorflow/efficientdet/lite2/detection/1"
detector = hub.load(model_url)

# -------------------- Object Detection --------------------
def detect_objects(image):
    input_tensor = tf.convert_to_tensor(image, dtype=tf.uint8)[tf.newaxis, ...]
    detections = detector(input_tensor)
    return detections[0]   # 👈 unwrap tuple


def draw_boxes(image, detections, threshold=0.3):
    # unwrap tuple → dict
    detections = detections[0]

    boxes = detections['detection_boxes'][0].numpy()
    scores = detections['detection_scores'][0].numpy()
    classes = detections['detection_classes'][0].numpy().astype(int)

    height, width, _ = image.shape

    for i in range(len(scores)):
        score = scores[i]
        class_id = classes[i]

        if score < threshold:
            continue

        # COCO knife class = 48
        if class_id != 48:
            continue

        y_min, x_min, y_max, x_max = boxes[i]
        x1, y1 = int(x_min * width), int(y_min * height)
        x2, y2 = int(x_max * width), int(y_max * height)

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(
            image,
            f"Knife {score:.2f}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )


# -------------------- Face Coverage --------------------
def calculate_face_coverage(face_landmarks, brightness):
    critical_landmarks = [33, 133, 362, 263, 1, 2, 98, 324, 13, 14]
    visible_landmarks = 0

    visibility_threshold = 0.6 - (brightness / 255.0) * 0.2
    z_min = -0.1 - (brightness / 255.0) * 0.05
    z_max = 0.1 + (brightness / 255.0) * 0.05

    for idx in critical_landmarks:
        lm = face_landmarks[idx]
        if lm.visibility > visibility_threshold and z_min < lm.z < z_max:
            visible_landmarks += 1

    return (visible_landmarks / len(critical_landmarks)) * 100

# -------------------- Behavior Detection --------------------
def analyze_behavior(pose_landmarks):
    left_wrist_y = pose_landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y
    right_wrist_y = pose_landmarks[mp_pose.PoseLandmark.RIGHT_WRIST].y
    nose_y = pose_landmarks[mp_pose.PoseLandmark.NOSE].y

    if left_wrist_y < nose_y or right_wrist_y < nose_y:
        return True
    return False

# -------------------- Video Capture --------------------
cap = cv2.VideoCapture(0)

alert_message = "Warning: Potential Threat!"
alert_duration = 3
last_alert_time = 0

# -------------------- Main Loop --------------------
while True:
    success, frame = cap.read()
    if not success:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness = np.mean(gray)

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    face_results = face_mesh.process(image_rgb)
    pose_results = pose.process(image_rgb)

    # ---- Face Coverage Alert ----
    if face_results.multi_face_landmarks:
        for face_landmarks in face_results.multi_face_landmarks:
            coverage = calculate_face_coverage(face_landmarks.landmark, brightness)
            print(f"Coverage: {coverage:.2f}%  Brightness: {brightness:.2f}")

            if coverage < 20:
                now = time.time()
                if now - last_alert_time > alert_duration:
                    cv2.putText(
                        frame,
                        alert_message,
                        (50, 50),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (0, 0, 255),
                        2
                    )
                    filename = f"warning_image_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.jpg"
                    cv2.imwrite(filename, frame)
                    last_alert_time = now

    # ---- Behavior Detection ----
    if pose_results.pose_landmarks:
        if analyze_behavior(pose_results.pose_landmarks.landmark):
            cv2.putText(
                frame,
                "Unusual Behavior Detected!",
                (50, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 0, 0),
                2
            )

    # ---- Object Detection ----
    detections = detect_objects(image_rgb)
    draw_boxes(frame, detections)

    cv2.imshow("Surveillance System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# -------------------- Cleanup --------------------
cap.release()
cv2.destroyAllWindows()
